<a href="https://colab.research.google.com/github/Muskan-Tarafder/BookBot/blob/main/LLM_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U bitsandbytes --break-system-packages
!pip install -q -U transformers accelerate peft trl datasets optimum --break-system-packages


In [2]:
!pip install -U bitsandbytes>=0.46.1 --break-system-packages

In [3]:
!pip install -q -U pyarrow --break-system-packages

In [4]:
from huggingface_hub import logout
logout()

Not logged in!


Loging in To HuggingFace with token

In [4]:
from huggingface_hub import notebook_login,login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from peft import LoraConfig, AutoPeftModelForCausalLM, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

Collection of database (5000 Samples) and creating Prompts

In [6]:
data = load_dataset(
    "swayista/book-recommender-dataset",
    data_files="data/books_with_categories.csv",
    split="train"
)
data_df = data.to_pandas()
data_df = data_df.fillna("")
data_df = data_df[:5000]

data_df["text"] = data_df.apply(lambda x:
    "###Human: Recommend a book about " + str(x["categories"]) +
    " ###Assistant: I recommend \"" + str(x["title"]) +
    "\" by " + str(x["authors"]) +
    ". " + str(x["description"]) +
    " It has an average rating of " + str(x["average_rating"]) + "/5.",
    axis=1
)

data = Dataset.from_pandas(data_df)

Model Selection and Settings

In [7]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training of Model

In [8]:
peft_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

In [9]:
from trl import SFTConfig, SFTTrainer
sft_config = SFTConfig(
    output_dir="tinyllama-finetuned-books",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    logging_steps=10,
    num_train_epochs=2,
    max_steps=300,
    save_steps=50,
    fp16=True,
    bf16=False,
    push_to_hub=False,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

trainer.train()


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.562441
20,2.328867
30,2.097968
40,1.992894
50,1.936819
60,1.923392
70,1.866670
80,1.912580
90,1.863307
100,1.897350


TrainOutput(global_step=300, training_loss=1.9165268707275391, metrics={'train_runtime': 828.6566, 'train_samples_per_second': 5.793, 'train_steps_per_second': 0.362, 'total_flos': 1.0380972199084032e+16, 'train_loss': 1.9165268707275391})

In [9]:
# Refining Model

from trl import SFTConfig, SFTTrainer
new_sft_config = SFTConfig(
    output_dir="tinyllama-finetuned-books",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=2,
    save_steps=50,
    fp16=True,
    bf16=False,
    push_to_hub=False,
    packing=False,
)

new_trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    peft_config=peft_config,
    args=new_sft_config,
    processing_class=tokenizer,
)

new_trainer.train()


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,2.179289
100,1.890753
150,1.879690


Step,Training Loss
50,2.179289
100,1.890753
150,1.879690
200,1.839509
250,1.826040
300,1.856278
350,1.846991
400,1.808464
450,1.841513
500,1.826382


TrainOutput(global_step=626, training_loss=1.872082408624716, metrics={'train_runtime': 1705.6175, 'train_samples_per_second': 5.863, 'train_steps_per_second': 0.367, 'total_flos': 2.141729611215667e+16, 'train_loss': 1.872082408624716})

In [11]:
!pip show trl | grep Version

Version: 0.29.0


Saving the model

In [12]:
# trial save

from google.colab import drive
drive.mount('/content/drive')

# Save model locally first
new_trainer.save_model("/content/tinyllama-finetuned-books-v2")
tokenizer.save_pretrained("/content/tinyllama-finetuned-books-v2")
print("✅ Saved locally!")

# Copy to Drive
!cp -r /content/tinyllama-finetuned-books-v2 /content/drive/MyDrive/

# Verify
!ls /content/drive/MyDrive/tinyllama-finetuned-books-v2

# Zip and copy to Drive
!zip -r tinyllama-finetuned-books-v2.zip tinyllama-finetuned-books-v2
!cp tinyllama-finetuned-books-v2.zip /content/drive/MyDrive/

# Verify everything
!ls /content/tinyllama-finetuned-books-v2
!ls /content/drive/MyDrive/tinyllama-finetuned-books-v2
!ls /content/drive/MyDrive/ | grep tinyllama

# Push to HuggingFace Hub as v2
model.push_to_hub("MuskanTara/tinyllama-finetuned-books-v2")
tokenizer.push_to_hub("MuskanTara/tinyllama-finetuned-books-v2")
print("✅ Pushed to HuggingFace Hub!")

Mounted at /content/drive
✅ Saved locally!
adapter_config.json	   README.md		  training_args.bin
adapter_model.safetensors  tokenizer_config.json
chat_template.jinja	   tokenizer.json
  adding: tinyllama-finetuned-books-v2/ (stored 0%)
  adding: tinyllama-finetuned-books-v2/adapter_config.json (deflated 57%)
  adding: tinyllama-finetuned-books-v2/tokenizer.json (deflated 85%)
  adding: tinyllama-finetuned-books-v2/tokenizer_config.json (deflated 44%)
  adding: tinyllama-finetuned-books-v2/README.md (deflated 65%)
  adding: tinyllama-finetuned-books-v2/training_args.bin (deflated 53%)
  adding: tinyllama-finetuned-books-v2/adapter_model.safetensors (deflated 8%)
  adding: tinyllama-finetuned-books-v2/chat_template.jinja (deflated 60%)
adapter_config.json	   README.md		  training_args.bin
adapter_model.safetensors  tokenizer_config.json
chat_template.jinja	   tokenizer.json
adapter_config.json	   README.md		  training_args.bin
adapter_model.safetensors  tokenizer_config.json
chat_templat

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e9v3lc1/model.safetensors:   0%|          | 1.17MB / 4.41GB            

README.md: 0.00B [00:00, ?B/s]

✅ Pushed to HuggingFace Hub!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/tinyllama-finetuned-books /content/drive/MyDrive/ # ✅ Fixed folder name


Mounted at /content/drive


In [ ]:
!cp -r /content/tinyllama-finetuned-books /content/drive/MyDrive/

In [ ]:
!ls /content/drive/MyDrive/tinyllama-finetuned-books

adapter_config.json	   checkpoint-100	  tokenizer.json
adapter_model.safetensors  README.md		  training_args.bin
chat_template.jinja	   tokenizer_config.json


In [ ]:
!ls /content

'=0.46.1'   drive   sample_data   tinyllama-finetuned-books


In [ ]:
!zip -r tinyllama-finetuned-books.zip tinyllama-finetuned-books
!cp tinyllama-finetuned-books.zip /content/drive/MyDrive/

  adding: tinyllama-finetuned-books/ (stored 0%)
  adding: tinyllama-finetuned-books/checkpoint-100/ (stored 0%)
  adding: tinyllama-finetuned-books/checkpoint-100/training_args.bin (deflated 53%)
  adding: tinyllama-finetuned-books/checkpoint-100/trainer_state.json (deflated 68%)
  adding: tinyllama-finetuned-books/checkpoint-100/scheduler.pt (deflated 62%)
  adding: tinyllama-finetuned-books/checkpoint-100/adapter_config.json (deflated 57%)
  adding: tinyllama-finetuned-books/checkpoint-100/scaler.pt (deflated 64%)
  adding: tinyllama-finetuned-books/checkpoint-100/tokenizer.json (deflated 85%)
  adding: tinyllama-finetuned-books/checkpoint-100/chat_template.jinja (deflated 60%)
  adding: tinyllama-finetuned-books/checkpoint-100/rng_state.pth (deflated 26%)
  adding: tinyllama-finetuned-books/checkpoint-100/optimizer.pt (deflated 8%)
  adding: tinyllama-finetuned-books/checkpoint-100/tokenizer_config.json (deflated 44%)
  adding: tinyllama-finetuned-books/checkpoint-100/README.md (de

In [ ]:
!ls /content/tinyllama-finetuned-books

checkpoint-100	README.md


In [ ]:
!ls /content/drive/MyDrive/tinyllama-finetuned-books

adapter_config.json	   checkpoint-100	  tokenizer.json
adapter_model.safetensors  README.md		  training_args.bin
chat_template.jinja	   tokenizer_config.json


In [ ]:
!ls /content/drive/MyDrive/ | grep tinyllama

tinyllama-finetuned-books
tinyllama-finetuned-books.zip


In [ ]:
trainer.save_model("tinyllama-finetuned-books")
tokenizer.save_pretrained("tinyllama-finetuned-books")

('tinyllama-finetuned-books/tokenizer_config.json',
 'tinyllama-finetuned-books/chat_template.jinja',
 'tinyllama-finetuned-books/tokenizer.json')

Push To HuggingFace Hub

In [ ]:
model.push_to_hub("MuskanTara/tinyllama-finetuned-books")
tokenizer.push_to_hub("MuskanTara/tinyllama-finetuned-books")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   9%|9         |  814kB / 9.02MB            

CommitInfo(commit_url='https://huggingface.co/MuskanTara/tinyllama-finetuned-books/commit/7b29b0f210572d75fff043528233cd7d355ba698', commit_message='Upload tokenizer', commit_description='', oid='7b29b0f210572d75fff043528233cd7d355ba698', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MuskanTara/tinyllama-finetuned-books', endpoint='https://huggingface.co', repo_type='model', repo_id='MuskanTara/tinyllama-finetuned-books'), pr_revision=None, pr_num=None)

# **Recommendation testing**

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, GenerationConfig
import torch, time

model_path = "/content/drive/MyDrive/tinyllama-finetuned-books"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="cuda"
)

generation_config = GenerationConfig(
    do_sample=True,
    top_k=1,
    temperature=0.1,
    max_new_tokens=200,
    pad_token_id=tokenizer.eos_token_id
)

inputs = tokenizer(
    "<|system|>You are a book recommendation assistant.</s><|user|>Recommend a book about science fiction</s><|assistant|>",
    return_tensors="pt"
).to("cuda")

st = time.time()
outputs = model.generate(**inputs, generation_config=generation_config)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print(f"Time: {time.time()-st:.2f}s")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|>You are a book recommendation assistant.<|user|>Recommend a book about science fiction<|assistant|>Here's a recommendation for a book about science fiction:

"The Last Days of California" by James Rollins

This is the third book in the "Sigma Force" series, which follows the adventures of a team of scientists and special agents who are tasked with stopping a deadly virus that threatens to wipe out the entire state of California. The book is a thrilling blend of science fiction and action, with a plot that keeps readers on the edge of their seats. It's a great choice for fans of James Rollins and the Sigma Force series.
Time: 6.02s


In [14]:
# CELL 1: Install new package
!pip install -q google-genai --break-system-packages

In [15]:
!pip install groq -q --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.6 MB/s eta 0:00:00


Integration with Groq API

In [21]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="/content/drive/MyDrive/tinyllama-finetuned-books-v2",
    repo_id="MuskanTara/tinyllama-finetuned-books-v2",
    repo_type="model"
)
print("✅ All files uploaded!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  61%|######1   | 5.53MB / 9.02MB            

  ...ooks-v2/training_args.bin:  14%|#3        |   776B / 5.58kB            

✅ All files uploaded!


In [22]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("MuskanTara/tinyllama-finetuned-books-v2")
print(list(files))

['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [23]:
from groq import Groq
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, GenerationConfig
import torch

# ============================================================
# Load your fine-tuned HuggingFace model
# ============================================================
model_path = "MuskanTara/tinyllama-finetuned-books-v2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
book_model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
generation_config = GenerationConfig(
    do_sample=True,
    top_k=50,
    temperature=0.7,
    max_new_tokens=200,
    pad_token_id=tokenizer.eos_token_id
)

# ============================================================
# Setup Groq for general conversation
# ============================================================
groq_client = Groq(api_key="GROQ_API_KEY")

# ============================================================
# Book keyword detector
# ============================================================
BOOK_KEYWORDS = ["book", "recommend", "read", "author", "novel", "fiction",
                 "horror", "comedy", "thriller", "fantasy", "romance", "genre",
                 "suggest", "story", "stories", "literature", "sci-fi", "mystery"]

def is_book_query(text):
    return any(word in text.lower() for word in BOOK_KEYWORDS)

history = []

# ============================================================
# Chat router
# ============================================================
def chat(user_query):
    if is_book_query(user_query):
        # Using fine-tuned HuggingFace model
        history.append(f"<|user|>{user_query}</s>")
        recent_history = history[-6:]
        prompt = (
            "<|system|>You are a book recommendation assistant. "
            "Recommend real books with title, author, and a short description.</s>"
            + "".join(recent_history)
            + "<|assistant|>"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = book_model.generate(**inputs, generation_config=generation_config)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        reply = response.split("<|assistant|>")[-1].strip()
        history.append(f"<|assistant|>{reply}</s>")
        return f"{reply}"
    else:
        # Using Groq for everything else
        groq_history = [
            {"role": "system", "content": (
                "You are a friendly assistant for a book recommendation chatbot. "
                "Keep responses short and friendly. "
                "Nudge the user to ask for book recommendations when relevant."
            )}
        ] + [
            {"role": "user" if i % 2 == 0 else "assistant", "content": m.split(">")[-1].replace("</s", "").strip()}
            for i, m in enumerate(history[-6:])
        ] + [{"role": "user", "content": user_query}]

        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=groq_history,
            max_tokens=150
        )
        reply = response.choices[0].message.content
        history.append(f"<|user|>{user_query}</s>")
        history.append(f"<|assistant|>{reply}</s>")
        return f"{reply}"

# ============================================================
# Chat loop
# ============================================================
print("📚 Book Recommender Chatbot (type 'quit' to exit, 'reset' to clear history)\n")
while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break
    if user_input.lower() == "reset":
        history.clear()
        print("History cleared!\n")
        continue
    print(f"Bot: {chat(user_input)}\n")

adapter_config.json:   0%|          | 0.00/988 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/9.02M [00:00<?, ?B/s]

📚 Book Recommender Chatbot (type 'quit' to exit, 'reset' to clear history)

You: hi
Bot: Hello! Welcome to our book chat. How's your day going so far? Got a particular type of book in mind, or do you need some recommendations?

You: something like calm and composed
Bot: You're looking for a book that evokes a feeling of calmness and composure. Reminds me of some great works of nature-inspired literature or soothing self-help books.

Would you like some book recommendations that fit this vibe?

You: yes
Bot: I'm excited to help you find your next great read! What genre are you in the mood for? Would you like some book recommendations?

You: calm
Bot: A calm and peaceful vibe! Are you looking for books to help you relax or find some tranquility? I'd be happy to recommend some soothing reads.

You: recommend some calm and composed books


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Sure thing! Here are some books that promise a restful and peaceful reading experience. 1. **The Art of Being Normal** by Lisa Williamson (It is a very easy and peaceful book to read. You could read it in one sitting. It is about a group of twelve-year-olds who have lived normal lives, but who suddenly find themselves suddenly living normal lives in an alternate world. It is a very calming and peaceful book. It's the kind of book that puts you in the middle of the action and you're not even aware that there is an entire alternate world going on around you. It is an excellent book for those who want a restful and peaceful reading experience. 2. **The Hundred-Year-Old Man Who Climbed Out of the Window and Disappeared** by Jonas Jonasson (It is a calming and peaceful book that can help you relax and unwind. It

You: quit


In [ ]:
from google import genai

client = genai.Client(api_key="API_KEY")

# List all available models
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/ge

# **API For** **Django** **Server**

In [24]:
!pip install flask pyngrok peft transformers torch -q
!pip install -q torch==2.1.0 transformers peft flask pyngrok --break-system-packages
# Set your ngrok auth token (free at https://ngrok.com)
!ngrok authtoken ngrok_api_key

from flask import Flask, request, jsonify
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, GenerationConfig
from pyngrok import ngrok
import torch

app = Flask(__name__)

print("Loading model...")
model_path = "MuskanTara/tinyllama-finetuned-books-v2"
hf_tokenizer = AutoTokenizer.from_pretrained(model_path)
hf_model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded!")

generation_config = GenerationConfig(
    do_sample=True,
    top_k=50,
    temperature=0.7,
    max_new_tokens=1000,
    pad_token_id=hf_tokenizer.eos_token_id
)

BOOK_KEYWORDS = ["book", "recommend", "read", "author", "novel", "fiction",
                 "horror", "comedy", "thriller", "fantasy", "romance", "genre",
                 "suggest", "story", "literature", "sci-fi", "mystery"]

def is_book_query(text):
    return any(word in text.lower() for word in BOOK_KEYWORDS)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    user_message = data.get('message', '')
    history = data.get('history', [])

    if not is_book_query(user_message):
        return jsonify({'reply': None, 'use_groq': True})

    hf_history = "".join([f"<|{m['role']}|>{m['content']}</s>" for m in history[-6:]])
    prompt = (
        "<|system|>You are a book recommendation assistant. "
        "Recommend only and only real books with title, author, and description. If not present then recommend some different real book from the genre.</s>"
        + hf_history
        + f"<|user|>{user_message}</s><|assistant|>"
    )

    inputs = hf_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = hf_model.generate(**inputs, generation_config=generation_config)
    reply = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    reply = reply.split("<|assistant|>")[-1].strip()

    return jsonify({'reply': reply, 'use_groq': False})

# Start ngrok
public_url = ngrok.connect(5000)
print(f"\n API running at: {public_url}/predict")
print("Copy this URL and paste it in services.py")

app.run(port=5000)

ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.1.0
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded!

 API running at: NgrokTunnel: "https://juliane-hyperactive-exaltedly.ngrok-free.dev" -> "http://localhost:5000"/predict
Copy this URL and paste it in services.py
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/Mar/2026 17:15:49] "POST /predict HTTP/1.1" 200 -
Both `max_new_tokens` (=1000) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [11/Mar/2026 17:16:25] "POST /predict HTTP/1.1" 200 -
Both `max_new_tokens` (=1000) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [11/Mar/2026 17:17:08] "POST /predict HTTP/1.1" 200 -
Both `max_new_tokens` (=1000) and `max_length`

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, GenerationConfig
import torch

model_path = "MuskanTara/tinyllama-finetuned-books"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="cuda"
)

generation_config = GenerationConfig(
    do_sample=True,
    top_k=50,
    temperature=0.7,
    max_new_tokens=200,
    pad_token_id=tokenizer.eos_token_id
)

# ✅ Detect if query is book-related or not
BOOK_KEYWORDS = ["book", "recommend", "read", "author", "novel", "fiction",
                 "horror", "comedy", "thriller", "fantasy", "romance", "genre"]

def is_book_query(text):
    return any(word in text.lower() for word in BOOK_KEYWORDS)

history = []

def recommend(user_query):
    # ✅ Handle greetings/off-topic without calling model
    if not is_book_query(user_query):
        greetings = ["hi", "hello", "hey", "thanks", "thank you", "ok", "okay", "bye"]
        if any(g in user_query.lower() for g in greetings):
            return "Hello! 👋 I'm your book recommendation assistant. Ask me for book recommendations by genre, author, or topic!"
        return "I'm only able to help with book recommendations! Try asking something like 'Recommend a horror book' or 'Suggest a good fantasy novel'."

    history.append(f"<|user|>{user_query}</s>")
    recent_history = history[-6:]

    prompt = (
        "<|system|>You are a book recommendation assistant. "
        "Recommend real books with title, author, and a short description.</s>"
        + "".join(recent_history)
        + "<|assistant|>"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, generation_config=generation_config)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    reply = response.split("<|assistant|>")[-1].strip()
    history.append(f"<|assistant|>{reply}</s>")
    return reply

print("📚 Book Recommender Chatbot (type 'quit' to exit, 'reset' to clear history)\n")
while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break
    if user_input.lower() == "reset":
        history.clear()
        print("🔄 History cleared!\n")
        continue
    print(f"Bot: {recommend(user_input)}\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

📚 Book Recommender Chatbot (type 'quit' to exit, 'reset' to clear history)

You: Hello
Bot: Hello! 👋 I'm your book recommendation assistant. Ask me for book recommendations by genre, author, or topic!

You: hi
Bot: Hello! 👋 I'm your book recommendation assistant. Ask me for book recommendations by genre, author, or topic!

You: how are you
Bot: I'm only able to help with book recommendations! Try asking something like 'Recommend a horror book' or 'Suggest a good fantasy novel'.

You: fine i'll ask something
Bot: Hello! 👋 I'm your book recommendation assistant. Ask me for book recommendations by genre, author, or topic!

You: quit
